Creating student earning parquet with buckets

Make sure to get the earning_buckets.csv from the Google Drive
and Needs Most-Recent-Cohorts-Field-of-Study.csv, Most-Recent-Cohorts-Institution.csv, CW2024_prelim.xlsx from https://collegescorecard.ed.gov/data/

In [ ]:
import polars as pl

# Needs Most-Recent-Cohorts-Field-of-Study.csv, Most-Recent-Cohorts-Institution.csv, CW2024_prelim.xlsx from https://collegescorecard.ed.gov/data/

def main():
    print("Scanning CSV files...")

    horizons = ["1YR", "4YR", "5YR"]

    # Cols holding "PS" (privacy suppressed), must be read as String first
    suppressed_cols = (
        [f"EARN_MDN_{h}" for h in horizons]
        + [f"EARN_COUNT_WNE_{h}" for h in horizons]
        + ["IPEDSCOUNT1"]
    )

    # Scorecard writes literal "NA" in some cols, including UNITID
    nulls = ["NA", "NULL", ""]

    # Helper to turn suppressed text into numbers
    def clean_suppressed(col_name, dtype=pl.Float64):
        return (
            pl.col(col_name)
            .str.strip_chars()
            .replace(["PS", ""], None)
            .cast(dtype, strict=False)
        )

    # Showing better documented CA public universities for demo purposes
    systems = {
        "California State University": "CSU",
        "University of California": "UC",
    }
    ca = (
        pl.read_excel("data/raw/CW2024_prelim.xlsx", sheet_name="Crosswalk")
        .select(
            pl.col("IPEDSMatch").cast(pl.Int64, strict=False).alias("UNITID"),
            pl.col("f1sysnam").replace_strict(systems, default=None).alias("system"),
        )
        .drop_nulls()
        .unique(subset="UNITID")
    )

    # Institution metadata
    meta = pl.scan_csv(
        "data/raw/Most-Recent-Cohorts-Institution.csv",
        schema_overrides={c: pl.String for c in ["UGDS"]},
        null_values=nulls,
        infer_schema_length=10000,
    ).select(
        pl.col("UNITID").cast(pl.Int64),
        pl.col("CITY").alias("city"),
        pl.col("STABBR").alias("state"),
        pl.col("UGDS").cast(pl.Int32, strict=False).alias("undergrad_enrollment"),
    )

    df = pl.scan_csv(
        "data/raw/Most-Recent-Cohorts-Field-of-Study.csv",
        schema_overrides={c: pl.String for c in suppressed_cols + ["CIPCODE"]},
        null_values=nulls,
        infer_schema_length=10000,
    )

    df = (
        df.with_columns(pl.col("UNITID").cast(pl.Int64))
        .join(ca.lazy(), on="UNITID", how="inner")
        .join(meta, on="UNITID", how="left")
        .with_columns(
            # Keep as text, CIP codes can start with a zero (0110)
            pl.col("CIPCODE").str.zfill(4).alias("cipcode"),

            *[clean_suppressed(f"EARN_MDN_{h}").alias(f"earn_mdn_{h.lower()}")
              for h in horizons],
            *[clean_suppressed(f"EARN_COUNT_WNE_{h}", pl.Int32).alias(f"earn_count_{h.lower()}")
              for h in horizons],
            clean_suppressed("IPEDSCOUNT1", pl.Int32).alias("completions"),

            # Why a value is missing: "PS" = withheld for privacy, blank = not reported
            *[pl.when(pl.col(f"EARN_MDN_{h}") == "PS").then(pl.lit("suppressed"))
               .when(pl.col(f"EARN_MDN_{h}").is_null()).then(pl.lit("not_reported"))
               .otherwise(pl.lit("usable"))
               .cast(pl.Categorical).alias(f"status_{h.lower()}")
              for h in horizons],
        )
        .select(
            pl.col("UNITID").alias("unitid"),
            pl.col("INSTNM").alias("institution"),
            "system", "city", "state", "undergrad_enrollment",
            "cipcode",
            pl.col("CIPDESC").alias("program"),
            pl.col("CREDLEV").cast(pl.Int16).alias("credlev"),
            "completions",
            *[f"earn_mdn_{h.lower()}" for h in horizons],
            *[f"earn_count_{h.lower()}" for h in horizons],
            *[f"status_{h.lower()}" for h in horizons],
        )
        .sort(["system", "institution", "cipcode", "credlev"])
    )

    final_df = df.collect()
    final_df.write_parquet("/con/student-earnings.parquet", compression="zstd")

    print(f"Processed {final_df.height:,} records.")
    schema = final_df.schema
    for col_name, dtype in schema.items():
        print(f"{col_name}: {dtype}")


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd

earnings = pd.read_parquet("/content/student-earnings.parquet")
buckets = pd.read_csv("/content/earning_buckets.csv")

# Clean accidental spaces so program names match reliably
earnings["program_key"] = earnings["program"].str.strip()
buckets["program_key"] = buckets["program"].str.strip()

# Keep only the lookup columns, then add department to every earnings row
earnings = earnings.drop(columns="department", errors="ignore")

earnings_with_buckets = earnings.merge(
    buckets[["program_key", "department"]],
    on="program_key",
    how="left",       # keep every row from earnings
    validate="m:1"    # many earnings rows can map to one program bucket
)

earnings_with_buckets = earnings_with_buckets.drop(columns="program_key")

earnings_with_buckets.to_parquet(
    "/content/student-earnings-with-buckets.parquet",
    index=False
)

print("Unmatched rows:", earnings_with_buckets["department"].isna().sum())

Cleaning Student data

In [ ]:
needs_cleaning = pd.read_parquet("/content/student-earnings-with-buckets.parquet")

needs_cleaning = needs_cleaning.drop(columns=[
    "unitid",
    "system",
    "city",
    "state",
    "undergrad_enrollment",
    "cipcode",
    "credlev",
    "completions",
    "status_1yr",
    "status_4yr",
    "status_5yr"
])

needs_cleaning.to_parquet("/content/student-earnings-with-buckets.parquet", index=False)

5 big stats on College scorecard data

In [ ]:
import pandas as pd

count_cols = [
    "earn_mdn_1yr",
    "earn_mdn_4yr",
    "earn_mdn_5yr"
]

df = read_parquet("/content/student-earnings-with-buckets.parquet")

# Convert values to numeric. Keep missing values as NaN:
# Pandas excludes NaN values from mean, median, std, and quantiles.
for col in count_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

grouped = df.groupby(["institution", "department"])


stats_parts = []

for col in count_cols:
    stats = grouped[col].agg(
        mean="mean",
        median="median",
        std="std",
        quantile_25=lambda x: x.quantile(0.25),
        quantile_75=lambda x: x.quantile(0.75),
    )

    # Makes names such as mean_earn_count_1yr
    stats.columns = [f"{stat}_{col}" for stat in stats.columns]
    stats_parts.append(stats)

student_stats = (
    pd.concat(stats_parts, axis=1)
    .reset_index()
    .rename(columns={"institution": "Institution", "department": "department"})
)

student_stats.to_parquet(
    "/content/student_pay_stats.parquet",
    index=False
)

display(student_stats)

In [4]:
import pandas as pd
salaries = pd.read_parquet("/content/student-earnings.parquet")

salaries.columns

Index(['unitid', 'institution', 'system', 'city', 'state',
       'undergrad_enrollment', 'cipcode', 'program', 'credlev', 'completions',
       'earn_mdn_1yr', 'earn_mdn_4yr', 'earn_mdn_5yr', 'earn_count_1yr',
       'earn_count_4yr', 'earn_count_5yr', 'status_1yr', 'status_4yr',
       'status_5yr'],
      dtype='object')